# DSC207 From-Scratch End-to-End Lab

This notebook is a clean reconstruction of the actual DSC207 workflow. It does **not** load the existing `.pkl` artifacts and it does not call the project helper functions. You will write each stage yourself, inspect the intermediate arrays and plots, and only then move to the next stage.

The scientific goal is parameter estimation:

```text
real detector strain → candidate window → physical parameters
```
The complete real-world system also needs event detection. That is treated as a separate final stage because the current DSC207 parameter-estimation experiment starts with a known simulated event time.

## Working rules

- Start from an empty notebook state.
- Do not load `DSC207/artifacts/*.pkl`.
- Do not import functions from `ligo_pipeline` for the implementation tasks.
- You may use standard scientific libraries and the external libraries required by the actual method.
- Plot every major object before transforming it.
- Print shape, dtype, units, time span, and finite-value status at every boundary.
- Keep a written experiment log: seed, URLs, GPS times, parameters, filter settings, and failures.

For the first run, use short segments and a small number of injections. After the logic works, scale toward the production configuration.

# 0. The target pipeline

You will build this sequence:

```text
configuration
  ↓
GWOSC ingestion of raw H1 strain
  ↓
time-domain inspection
  ↓
FFT and frequency-domain inspection
  ↓
noise PSD / ASD estimation
  ↓
draw binary parameters
  ↓
generate IMRPhenomD waveform
  ↓
inject waveform into real noise
  ↓
calculate matched-filter SNR and accept/reject
  ↓
whiten, bandpass, and crop around merger
  ↓
create X and theta
  ↓
train SNPE with a MAF density estimator
  ↓
sample posterior and evaluate calibration
```

The central equation is $x(t)=n(t)+h(t;\theta)$: detector noise $n(t)$ plus a waveform $h(t;\theta)$ generated from physical parameters $\theta$.

## Task 0: Write the experiment specification

Before coding, write a small configuration dictionary or table containing: detector `H1`, observing run, sample rate, raw duration, crop duration, low/high analysis frequency, number of noise segments, number of injections, random seeds, and output paths.

Start with an educational configuration such as a 64- or 256-Hz toy signal if your environment cannot yet run PyCBC. Then write the production values beside it: 4096 Hz, 1024-second noise segments, 4-second crop, 35-350 Hz bandpass, and SNR threshold 8.

**Check:** explain which values change the data and which values only change bookkeeping.

# 1. Ingestion: obtain raw strain from GWOSC

**Concept.** Ingestion is the boundary between an external data source and your experiment. GWOSC provides public strain data. A strain segment is a sampled time series from a detector, with GPS start time and sample rate. It is not yet a training row and it has no mass or distance label.

For a segment beginning at GPS time $t_0$, sample rate $f_s$, and $N$ samples, the sample-time axis is $t_k=t_0+k/f_s$. The duration is approximately $N/f_s$.

In production, ingestion also records provenance: detector, observing run, GPS interval, source URL, download time, sample rate, and checksum or local filename.

## Task 1: Ingest one small real segment

Use a GWOSC client or its documented public URL to fetch one short H1 strain segment. Do not use any existing project pickle. Begin with a duration small enough to download and inspect. Store the returned strain array and metadata in your own local variables.

Print: detector, GPS start/end, sample rate, number of samples, duration, dtype, min/max/mean/std, and whether every value is finite. Plot the first few seconds in the time domain.

**Expected object:** one raw strain array plus a provenance dictionary.

**Failure tests:** deliberately request an invalid interval and test how your code reports an unavailable segment.

# 2. Time-domain foundations

A detector measures strain, a dimensionless fractional change in length. The recorded series is a sequence $x[0],...,x[N-1]$. The sampling interval is $\Delta t=1/f_s$.

The mean is $\bar{x}=N^{-1}\sum_n x[n]$. The variance measures fluctuation around the mean, $\sigma^2=N^{-1}\sum_n(x[n]-\bar{x})^2$. These are descriptive checks, not event labels.

Real detector noise is not just independent Gaussian noise. It can be nonstationary, contain lines, and have transient artifacts. Plotting is therefore a scientific check, not decoration.

## Task 2: Explore the raw time series

Create a time axis from the metadata. Plot the full downloaded segment, several zoomed windows, a histogram, and rolling mean/std or blockwise statistics. Mark any suspicious transients or changing noise scale.

**Expected output:** a short written description of whether the segment appears stationary enough for a toy experiment.

**Check:** compare your measured duration with $N/f_s$ and verify the time step numerically.

# 3. Frequency domain and FFT

The discrete Fourier transform represents the same finite signal as a sum of complex sinusoids:

$$X[k]=\sum_{n=0}^{N-1}x[n]e^{-2\pi i kn/N}.$$

For real-valued strain, use the one-sided real FFT. The frequency bin is $f_k=kf_s/N$, the Nyquist limit is $f_s/2$, and the frequency resolution is $\Delta f=f_s/N$. The magnitude $|X[k]|$ is an amplitude measure; it is not yet a PSD.

Windowing reduces spectral leakage caused by cutting a finite segment. A Hann window is common: multiply the time series by a smooth taper before the FFT, while remembering that normalization changes.

## Task 3: Compute and interpret the FFT

Implement the one-sided FFT twice: once without a window and once with a Hann window. Plot magnitude against frequency for the real strain. Mark the Nyquist frequency and the 35-350 Hz analysis band.

Repeat with a toy sine wave whose frequency you know exactly. Measure the peak error and explain spectral leakage and frequency resolution.

**Expected output:** time plot, spectrum plot, and a paragraph distinguishing amplitude spectrum from PSD.

# 4. PSD and ASD: what the noise contributes

The power spectral density $S_n(f)$ estimates noise power per unit frequency. Informally, it tells you where the detector is noisy. The amplitude spectral density is $\mathrm{ASD}(f)=\sqrt{S_n(f)}$.

A single FFT periodogram is noisy. Welch's method divides data into overlapping windows, computes periodograms, and averages them. This trades some frequency resolution for a more stable estimate.

The PSD is central to this project because it is used for whitening and matched-filter SNR. It is estimated from noise, not assigned astrophysical parameters.

## Task 4: Estimate a PSD yourself

Implement Welch-style PSD estimation using only NumPy first: choose segment length, overlap, window, FFT, periodogram normalization, and averaging. Then compare your result with a trusted SciPy implementation if available.

Plot PSD and ASD on log-log axes. Report frequency resolution, Nyquist frequency, and the frequency of the largest noise feature.

**Failure tests:** use a constant signal and a signal containing NaNs. Decide what your estimator should return.

# 5. Simulated binary-black-hole parameters

Real events do not provide exact labels for every physical parameter. To create supervised examples, choose parameters from explicit priors and generate a theoretical waveform.

The production parameter draw includes component masses $m_1,m_2$, aligned spins $\chi_1,\chi_2$, and luminosity distance $d$. The project enforces $m_1\ge m_2$. Distance is drawn with a volume-weighted rule, not uniformly in distance:

$$d = [u(d_{max}^3-d_{min}^3)+d_{min}^3]^{1/3},\quad u\sim U(0,1).$$

Derived labels include chirp mass $\mathcal{M}=(m_1m_2)^{3/5}/(m_1+m_2)^{1/5}$, mass ratio $q=m_2/m_1$, and an effective spin coordinate.

## Task 5: Draw and audit a prior table

Implement the parameter draw from scratch with a fixed seed. Generate 100 draws, enforce mass ordering, check every bound, compute derived parameters, and plot histograms. Compare uniform-distance and volume-weighted-distance draws so the difference is visible.

**Expected output:** a parameter table, bound checks, derived-label table, and prior plots.

**Check:** explain why the prior is part of the scientific experiment and why the model should not be evaluated outside the declared prior without a deliberate design.

# 6. Waveform generation

A waveform approximant maps physical parameters to polarizations $h_+(t)$ and $h_\times(t)$. The actual notebook uses PyCBC's `get_td_waveform` with the IMRPhenomD approximant. The waveform includes inspiral, merger, and ringdown behavior.

The generated waveform has its own sample rate, duration, time offset, and peak location. You must inspect these before adding it to noise. A waveform is not automatically aligned to the center of a noise segment.

The distance scales amplitude approximately inversely: farther systems are weaker. Masses and spins change phase evolution, duration, and merger morphology.

## Task 6: Generate and compare waveforms

Use PyCBC directly, not the project wrapper, to generate five IMRPhenomD waveforms. Plot plus polarization, cross polarization, and their amplitude envelope. Annotate each plot with masses, spins, distance, sample rate, duration, and peak index.

Change one physical parameter at a time and compare the waveforms. Record which changes mainly affect amplitude, phase evolution, or duration.

**Fallback:** if PyCBC is unavailable, complete the same experiment with a toy chirp and clearly label it as a toy, not an astrophysical waveform.

# 7. Injection: add the event to real noise

An injection creates a labeled noisy example:

$$x(t)=n(t)+h(t;\theta).$$

Choose a merger offset inside the noise segment. Find the waveform peak, align that peak to the merger index, and add the waveform samples into the noise buffer. Save the placement metadata: noise identifier, merger index, waveform start, and local merger index.

The resulting record contains noise plus an event. The event parameters are known because you generated the waveform. Pure noise has no mass or distance values.

## Task 7: Build one injection manually

Use one ingested noise segment and one generated waveform. Choose a merger time away from both boundaries. Add the waveform by hand using array indices. Plot noise-only, waveform-only, and injected strain on aligned axes, including a zoom around the merger.

Validate finiteness, bounds, and exact placement. Store your own record dictionary with `strain`, `params`, `merger_idx`, and provenance.

**Expected output:** a visibly inspectable injection and a record you can serialize later.

# 8. Matched-filter SNR and acceptance

SNR measures how detectable a waveform is relative to the detector noise. For a noise PSD $S_n(f)$, the optimal matched-filter SNR is conceptually

$$\rho^2=4\int_{f_{low}}^{f_{high}}\frac{|\tilde h(f)|^2}{S_n(f)}df.$$

The PSD weights frequencies: a waveform component in a quiet band contributes more than the same amplitude in a noisy band. In the actual project, PyCBC computes the optimal SNR and injections with $\rho<8$ are rejected.

SNR is an acceptance/quality statistic. It is not one of the four final target parameters.

## Task 8: Calculate and interpret SNR

First implement the integral as a frequency-bin sum using your waveform FFT and PSD. Then compare your result with PyCBC's `sigma` or matched-filter calculation. Plot SNR contribution by frequency.

Create injections across a range of distances and observe how SNR changes. Implement an acceptance rule and report attempted, accepted, below-threshold, and waveform-failure counts.

**Check:** explain why accepting only high-SNR examples changes the training distribution.

# 9. Whitening and normalization

Whitening uses the noise PSD to normalize frequency components. In one common convention:

$$\tilde x_w(f)=\frac{\tilde x(f)}{\sqrt{S_n(f)/2}},$$

followed by an inverse FFT. Exact constants depend on the FFT and PSD conventions, so match your convention consistently and verify behavior empirically.

This is not the same as ordinary ML standardization. Standardization might subtract a mean and divide by a global standard deviation. Whitening uses frequency-dependent noise power. After whitening, the signal is often additionally normalized or checked for finite values before filtering.

## Task 9: Whiten one injection

Estimate the PSD from a noise-only portion, interpolate it to the injection FFT frequency bins, apply your whitening formula, and inverse-transform. Plot raw and whitened time series plus their PSD/ASD estimates.

Add a documented PSD floor to prevent division by zero. Compare the variance by frequency before and after whitening.

**Expected output:** a visible reduction in the noise color, not a claim that the noise disappeared. Explain which normalization constants you used.

# 10. Bandpass filtering

The analysis band is a scientific choice. The production configuration uses 35-350 Hz. A bandpass suppresses frequencies below $f_{low}$ and above $f_{high}$, usually with a Butterworth filter.

A causal filter introduces phase delay. A zero-phase forward/backward filter avoids phase delay in offline analysis but uses information from both directions and can have edge effects. The actual processor uses a zero-phase Butterworth bandpass.

Filtering changes the signal. Always plot the frequency response and inspect edge behavior.

## Task 10: Design and test the bandpass

Design a fourth-order Butterworth bandpass with your chosen sample rate. Plot its frequency response. Apply it to whitened noise and to an injection. Compare time and frequency plots before and after.

Test a low-frequency sine, an in-band sine, and a high-frequency sine. Measure attenuation in each case. Inspect the first and last seconds for edge artifacts.

**Check:** explain why `high < sample_rate/2` is required.

# 11. Crop the merger-centered ML window

The injected buffer is larger than the model input. Once the merger index is known, crop a fixed duration around it. For a 4-second window at 4096 Hz:

$$N_{crop}=4\times4096=16384.$$

The crop is centered using metadata from the simulation. This is why the current project performs parameter estimation after a known candidate time rather than blind event detection.

## Task 11: Crop and validate

Crop one filtered injection around its merger index. Plot the full buffer with crop boundaries and the final crop alone. Verify exact length, finite values, and event position inside the crop.

Test a merger near the left edge and near the right edge. Decide whether to reject, pad, or move the candidate and document the scientific consequence.

**Expected output:** one fixed-shape model input and explicit boundary behavior.

# 12. Build the final parameter-estimation dataset

For each accepted injection, run the same preprocessing chain and append exactly one fixed-length row to `X`. Build one target row in `theta` using the derived parameters.

```text
X[i]     = processed 4-second noisy strain
theta[i] = [chirp_mass, mass_ratio, chi_eff, distance]
```

Pure noise is not assigned zero mass or distance. Pure noise belongs in a separate event-detection dataset, or it must have missing parameter targets with a masked loss in a deliberately designed multi-task model.

## Task 12: Generate a small dataset from scratch

Generate at least 20 accepted injections using your own ingestion, parameter draw, waveform, injection, SNR, whitening, filtering, and crop code. Keep an audit list for rejected attempts.

Assemble `X` and `theta`. Print shape, dtype, finite status, five target rows, five waveform summaries, and the parameter ranges actually present. Plot a grid of examples.

**Check:** verify that every accepted event contributes to both arrays or neither array.

# 13. Train/test design

A training pair is still ordinary supervised learning: `X_train[i]` is paired with `theta_train[i]`. The model must never fit on the test set.

The actual project uses O3a for training plus a 50-example holdout and O4a as an 800-example cross-run evaluation set. O4a is not another training set.

If overlapping windows from one noise segment are split randomly, leakage can occur because nearly identical noise appears in both sets. Prefer splitting by source segment, generated seed, or observing run when that matches your scientific question.

## Task 13: Split without leakage

Create a train/validation/test design for your generated examples. Record the source noise segment and random seed for every row. Compare a naive random split with a grouped split by source noise and explain the difference.

Do not tune preprocessing thresholds using the final test set. Write down exactly which arrays each training step can access.

**Expected output:** a data-flow diagram and a leakage audit.

# 14. Models used by the actual project

The production model is simulation-based inference using **SNPE** (Sequential Neural Posterior Estimation) with a **MAF** (Masked Autoregressive Flow) density estimator.

The model learns a conditional posterior $p(\theta\mid x)$ rather than only a single point prediction. A posterior can represent uncertainty and correlations between parameters.

The workflow is:

1. Define a prior over $\theta$.
2. Simulate many pairs $(\theta,x)$ using the injection pipeline.
3. Append simulations to SNPE.
4. Train a neural density estimator.
5. Build a posterior conditioned on an observed waveform.
6. Draw posterior samples and summarize them.

A simple MLP regressor is useful first as a baseline, but it does not provide the same posterior representation.

## Task 14: Train a baseline before SNPE

Train an ordinary multi-output regression baseline on your small `X/theta` dataset. Report parameter errors, plot predicted versus true values, and inspect residuals against SNR and distance.

Then write the exact conceptual difference between this baseline and SNPE. Do not install or run the full SNPE model until the baseline data contract is correct.

**Expected output:** a baseline report that tells you whether the data contains learnable information at all.

## Task 15: Train a small SNPE experiment

Only after the baseline works, use `sbi` with a small toy or small real-like dataset. Start with two targets if four targets are too expensive. Define the prior explicitly, append simulations, train SNPE with a MAF, condition on one held-out waveform, and draw posterior samples.

Plot marginal posterior histograms, pairwise parameter plots, and true values overlaid. Record package versions and seeds.

**Expected output:** posterior samples with shape `(number_of_samples, number_of_parameters)` and a clear explanation of what conditioning means.

# 16. Evaluation and calibration

There are two different questions:

- **Accuracy:** are the estimated parameters close to the known injected parameters?
- **Calibration:** do posterior uncertainty statements contain the truth at the advertised rate?

Simulation-Based Calibration (SBC) uses simulations with known truth. For each parameter, compare the true value with posterior samples and record its rank. Well-calibrated ranks should not show systematic concentration at the edges or center.

Also compare performance by SNR, distance, parameter region, source noise segment, and observing run.

## Task 16: Evaluate the complete experiment

For held-out examples, calculate point summaries and credible intervals from posterior samples. Plot coverage by nominal interval, posterior width versus SNR, error versus distance, and rank histograms.

Create one intentionally bad posterior or biased estimator and verify that your diagnostics can detect it. Record failed sampling attempts rather than silently deleting them.

**Expected output:** an accuracy report and a separate calibration report.

# 17. Event detection: the missing first stage

The current parameter-estimation experiment knows the simulated merger location so it can crop around it. A real search first scans data and decides whether an event candidate exists.

Detection dataset:

```text
noise only          → event_label = 0
noise + injection   → event_label = 1
```

Pure-noise rows have no mass, spin, or distance answer. Do not encode those physical parameters as zero unless a deliberately masked multi-task loss treats them as missing. A separate classifier is clearer for the first implementation.

## Task 17: Build a two-stage prototype

Create a balanced toy detection dataset from your own noise and injection generator. Train a classifier and report precision, recall, false positives, and false negatives. Then pass only predicted event windows to your parameter estimator.

Define the handoff contract: candidate GPS/time index, window boundaries, detector, confidence, and the waveform array passed to stage two.

**Expected output:** a complete diagram and a working toy detector-to-estimator handoff.

# 18. Final end-to-end capstone

Run a small complete experiment from scratch. Use one or two local GWOSC noise segments, generate a limited number of accepted injections, process them, create grouped train/test splits, train a baseline, and evaluate it. If dependencies permit, repeat with a small SNPE/MAF posterior.

Your final report must include:

1. Ingestion provenance and raw data plots.
2. Time-domain and frequency-domain plots.
3. PSD/ASD derivation and interpretation.
4. Parameter prior plots and waveform comparisons.
5. Injection plot with merger location.
6. SNR calculation and rejection counts.
7. Whitening, normalization, bandpass, and crop plots.
8. Final `X`/`theta` shapes and five records.
9. Train/test split and leakage reasoning.
10. Model definition and prediction plots.
11. Accuracy and calibration results.
12. What would be required to add real event detection.

# Final reference: what each object means

```text
raw strain       = detector measurement from GWOSC
noise segment     = real background with no astrophysical label
theta             = simulated physical parameters
waveform          = theoretical signal h(t; theta)
injection         = noise + waveform, with known theta
PSD               = noise power as a function of frequency
SNR               = detectability of a waveform relative to PSD
whitened strain   = frequency-normalized strain using the PSD
X                 = final fixed-length processed waveform
theta target      = [chirp_mass, mass_ratio, chi_eff, distance]
posterior         = probability distribution p(theta | X)
event detector    = separate model for event/no-event
```

The notebook is complete when you can follow one example through every line above and explain both its array shape and its scientific meaning.